# Notebook 02: Real KV Cache Memory Measurement vs. Formula

`[REAL]` Companion to Module 02. Real experiments on the RTX 4060 with `Qwen/Qwen2.5-0.5B-Instruct` -- a real model that itself uses **Grouped-Query Attention** (confirmed via its real config: 14 query heads, only 2 real KV heads), making this a direct, real test of Module 02's GQA memory formula, not just an MHA illustration.

**Revised measurement approach (per signed-off plan):** `torch.cuda.memory_allocated()` alone is *not* used as a stand-in for KV-cache memory -- it includes weight memory, activations, and other allocations. This notebook instead (1) computes the real, exact KV-cache tensor memory directly from `past_key_values` tensor shapes (ground truth, not an allocator-level proxy), and (2) separately reports real `memory_allocated()` vs. `memory_reserved()` deltas across sequence lengths to directly expose real PyTorch CUDA-allocator overhead.

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = AutoConfig.from_pretrained(MODEL_NAME)
N_LAYERS = config.num_hidden_layers
N_KV_HEADS = config.num_key_value_heads
N_Q_HEADS = config.num_attention_heads
D_HEAD = config.hidden_size // config.num_attention_heads
print(f"Real config: n_layers={N_LAYERS}, n_query_heads={N_Q_HEADS}, n_kv_heads={N_KV_HEADS}, d_head={D_HEAD}")
print(f"This model IS a real GQA model: {N_KV_HEADS} KV heads share across {N_Q_HEADS} query heads "
      f"({N_Q_HEADS // N_KV_HEADS}x sharing ratio)")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16).to(DEVICE)
model.eval()
print(f"\nLoaded {MODEL_NAME} at FP16 on {DEVICE}")

D:\Study\Prep\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Real config: n_layers=24, n_query_heads=14, n_kv_heads=2, d_head=64
This model IS a real GQA model: 2 KV heads share across 14 query heads (7x sharing ratio)


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:  89%|████████▊ | 257/290 [00:00<00:00, 2560.92it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 2656.63it/s]


Loaded Qwen/Qwen2.5-0.5B-Instruct at FP16 on cuda


## 1. Module 02's Formula, Applied to This Model's Real Config

`[COMPUTED FROM REAL DATA]` Module 02's formula: $\text{Mem}_{KV} = 2 \times B \times L \times N_{layers} \times N_{KV\_heads} \times d_{head} \times \text{bytes}_{dtype}$, evaluated with this model's own real, extracted config values (not the illustrative 7B numbers used in the module itself).

In [2]:
def kv_cache_formula_bytes(batch_size, seq_len, n_layers=N_LAYERS, n_kv_heads=N_KV_HEADS,
                            d_head=D_HEAD, bytes_per_elem=2):
    return 2 * batch_size * seq_len * n_layers * n_kv_heads * d_head * bytes_per_elem

SEQ_LENGTHS = [128, 512, 1024, 2048]
BATCH_SIZE = 1

for L in SEQ_LENGTHS:
    predicted_bytes = kv_cache_formula_bytes(BATCH_SIZE, L)
    print(f"L={L:5d}: formula-predicted KV cache = {predicted_bytes:,} bytes ({predicted_bytes/1024/1024:.3f} MB)")

print("\n(pending real measured comparison)")

L=  128: formula-predicted KV cache = 1,572,864 bytes (1.500 MB)
L=  512: formula-predicted KV cache = 6,291,456 bytes (6.000 MB)
L= 1024: formula-predicted KV cache = 12,582,912 bytes (12.000 MB)
L= 2048: formula-predicted KV cache = 25,165,824 bytes (24.000 MB)

(pending real measured comparison)


## 2. Real Ground-Truth Measurement: Exact KV-Cache Tensor Memory from `past_key_values`

`[REAL]` Running a real forward pass with `use_cache=True` at each real sequence length and summing the exact real byte size of every returned key/value tensor (`numel() * element_size()`, summed across all layers) -- a direct, exact measurement of the real KV-cache tensor memory PyTorch actually allocated for it, not an allocator-level proxy.

In [3]:
def extract_kv_tensors(past_kv):
    """Version-robust extraction across transformers' evolving Cache API."""
    if hasattr(past_kv, "layers"):  # transformers >=4.5x: Cache.layers[i].keys/.values
        tensors = []
        for layer in past_kv.layers:
            tensors.append(layer.keys)
            tensors.append(layer.values)
        return tensors
    if hasattr(past_kv, "key_cache"):  # older Cache API: separate key_cache/value_cache lists
        return list(past_kv.key_cache) + list(past_kv.value_cache)
    return [t for layer in past_kv for t in layer]  # legacy tuple-of-tuples API

def real_kv_cache_tensor_bytes(seq_len, batch_size=1):
    input_ids = torch.randint(0, tokenizer.vocab_size, (batch_size, seq_len), device=DEVICE)
    with torch.no_grad():
        outputs = model(input_ids, use_cache=True)
    tensors = extract_kv_tensors(outputs.past_key_values)
    total_bytes = sum(t.numel() * t.element_size() for t in tensors)
    del outputs
    return total_bytes

ground_truth_results = []
for L in SEQ_LENGTHS:
    torch.cuda.empty_cache() if DEVICE == "cuda" else None
    measured_bytes = real_kv_cache_tensor_bytes(L, BATCH_SIZE)
    predicted_bytes = kv_cache_formula_bytes(BATCH_SIZE, L)
    ground_truth_results.append({"seq_len": L, "measured_bytes": measured_bytes, "predicted_bytes": predicted_bytes})
    match = "EXACT MATCH" if measured_bytes == predicted_bytes else f"diff={measured_bytes - predicted_bytes:+,} bytes"
    print(f"L={L:5d}: measured={measured_bytes:,} bytes, predicted={predicted_bytes:,} bytes -- {match}")

print("\n(pending real interpretation)")

L=  128: measured=1,572,864 bytes, predicted=1,572,864 bytes -- EXACT MATCH
L=  512: measured=6,291,456 bytes, predicted=6,291,456 bytes -- EXACT MATCH
L= 1024: measured=12,582,912 bytes, predicted=12,582,912 bytes -- EXACT MATCH


L= 2048: measured=25,165,824 bytes, predicted=25,165,824 bytes -- EXACT MATCH

(pending real interpretation)


**Real result: exact match at all 4 sequence lengths.** `128` → `1,572,864` bytes measured vs. `1,572,864` predicted; `512` → `6,291,456` vs. `6,291,456`; `1024` → `12,582,912` vs. `12,582,912`; `2048` → `25,165,824` vs. `25,165,824` — `EXACT MATCH` at every single point. This is a real, direct, byte-for-byte confirmation of Module 02's formula (adapted with this real GQA model's own `n_layers=24`, `n_kv_heads=2`, `d_head=64`) against genuine executed-model tensor sizes, not an approximation.

## 3. Real CUDA Allocator Overhead: Allocated vs. Reserved Memory

`[REAL]` Separately from the exact tensor-level measurement above, this section measures real `torch.cuda.memory_allocated()` and `torch.cuda.memory_reserved()` *deltas* across real sequence lengths -- isolating KV-cache-attributable growth from the fixed real weight/activation baseline, and directly exposing the real, honest gap between what's allocated and what the CUDA caching allocator has reserved (a real, distinct overhead source from the exact tensor size measured in Section 2).

In [4]:
if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    baseline_allocated = torch.cuda.memory_allocated()
    baseline_reserved = torch.cuda.memory_reserved()
    print(f"Baseline (model loaded, no forward pass yet): allocated={baseline_allocated:,} bytes, "
          f"reserved={baseline_reserved:,} bytes")

    allocator_results = []
    for L in SEQ_LENGTHS:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        pre_allocated = torch.cuda.memory_allocated()
        input_ids = torch.randint(0, tokenizer.vocab_size, (BATCH_SIZE, L), device=DEVICE)
        with torch.no_grad():
            outputs = model(input_ids, use_cache=True)
        post_allocated = torch.cuda.memory_allocated()
        post_reserved = torch.cuda.memory_reserved()
        allocated_delta = post_allocated - pre_allocated
        reserved_minus_allocated = post_reserved - post_allocated
        allocator_results.append({
            "seq_len": L, "allocated_delta": allocated_delta,
            "reserved": post_reserved, "allocated": post_allocated,
            "reserved_minus_allocated": reserved_minus_allocated,
        })
        print(f"L={L:5d}: allocated_delta={allocated_delta:,} bytes, "
              f"reserved-allocated gap={reserved_minus_allocated:,} bytes")
        del outputs
else:
    print("CUDA not available -- skipping allocator-level measurement.")
    allocator_results = []

print("\n(pending real interpretation)")

Baseline (model loaded, no forward pass yet): allocated=1,004,841,984 bytes, reserved=1,061,158,912 bytes
L=  128: allocated_delta=41,419,776 bytes, reserved-allocated gap=61,034,496 bytes
L=  512: allocated_delta=161,876,992 bytes, reserved-allocated gap=93,668,352 bytes
L= 1024: allocated_delta=323,751,936 bytes, reserved-allocated gap=214,905,856 bytes


L= 2048: allocated_delta=648,028,160 bytes, reserved-allocated gap=666,571,776 bytes

(pending real interpretation)


## 4. Real Interpretation: Why `memory_allocated()` Alone Is a Poor KV-Cache Proxy

`[REAL]` The real measured `allocated_delta` is dramatically larger than Section 2's exact KV-cache tensor size — `41,419,776` bytes measured vs. only `1,572,864` bytes of real KV cache at `L=128` (a `26.33x` gap), and the ratio stays close to `25.7x`-`25.8x` at `L=512`, `L=1024`, and `L=2048` too. This is a real, direct, quantitative demonstration of exactly why the signed-off Track 2 plan ruled out `memory_allocated()` alone as a KV-cache proxy.

**Real root cause, identified and verified:** this model's vocabulary is `151,936` tokens. A single (non-cached-generation) forward pass computes real logits of shape `(batch, seq_len, vocab_size)` at FP16 — for `L=128`, that's `1 × 128 × 151,936 × 2 bytes ≈ 38,895,616` bytes, which is `93.9%` of the real measured `41,419,776`-byte delta; at `L=512/1024/2048` the logits tensor accounts for `96.0%`-`96.1%` of the real measured delta. The real KV cache itself is a small minority of what `memory_allocated()`'s delta actually captures — the logits tensor (an activation, scaling with `seq_len × vocab_size`, unrelated to Module 02's KV-cache formula) dominates it almost entirely in this single-forward-pass measurement setup.

**Allocated vs. reserved gap (the other half of the revised methodology):** the real `reserved - allocated` gap also grows with sequence length — `61,034,496` bytes at `L=128` up to `666,571,776` bytes at `L=2048` — a real, direct measurement of PyTorch's CUDA caching allocator reserving more memory than is strictly allocated at any instant, a genuine, distinct overhead source from both the KV-cache tensor size (Section 2) and the logits-dominated allocation delta (above).

**Bottom line:** Module 02's formula is exactly correct as a prediction of real KV-cache tensor memory (Section 2's `EXACT MATCH` at all 4 points) — but real, live GPU memory usage during a forward pass reflects several real, separate, additive sources (weights, KV cache, activations like logits, and allocator reservation slack) that a single `memory_allocated()` reading conflates together, exactly as the revised plan anticipated.